# Phase 5: Spatial Feature Engineering

This notebook builds the spatial and preserved transaction features required for model training.

Scope:
- load merged transaction-property data
- load processed road and facilities layers
- create nearest-distance features
- create facility counts
- create centroid latitude/longitude
- save final training dataset and feature summary


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import geopandas as gpd

from backend.src.config import configure_logging, ensure_directories, settings
from backend.src.feature_engineering import build_model_training_dataset, save_model_training_dataset

configure_logging()
ensure_directories()
PROJECT_ROOT, settings


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'),
 Settings(project_root=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc'), raw_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data'), interim_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/interim'), processed_data_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/processed'), reports_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports'), models_dir=WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/models'), transaction_file='tran_data.xlsx', property_shapefile='ai_usecase_data240326.shp', road_shapefile='ai_usecase_road240326.shp', facility_shapefile='ai_usecase_facilities240326.shp', log_level='INFO'))

In [2]:
merged_gdf = gpd.read_parquet(settings.interim_data_dir / 'transactions_property_merged.parquet')
roads_gdf = gpd.read_parquet(settings.interim_data_dir / 'roads_gis.parquet')
facilities_gdf = gpd.read_parquet(settings.interim_data_dir / 'facilities_gis.parquet')

merged_gdf.shape, roads_gdf.shape, facilities_gdf.shape


((273587, 79), (10345, 10), (238, 21))

In [3]:
{
    'merged_crs': str(merged_gdf.crs),
    'roads_crs': str(roads_gdf.crs),
    'facilities_crs': str(facilities_gdf.crs),
    'rows_with_geometry': int(merged_gdf.geometry.notna().sum()),
}


{'merged_crs': '{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "WGS 84 / UTM zone 45N", "base_crs": {"name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "

In [4]:
training_gdf, summary = build_model_training_dataset(merged_gdf, roads_gdf, facilities_gdf)
summary


FeatureSummary(input_row_count=273587, output_row_count=273587, projected_crs='{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "WGS 84 / UTM zone 45N", "base_crs": {"name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "de

In [5]:
training_gdf[['distance_to_nearest_road', 'distance_to_nearest_facility', 'facility_count_500m', 'facility_count_1km', 'latitude', 'longitude']].head()


,distance_to_nearest_road,distance_to_nearest_facility,facility_count_500m,facility_count_1km,latitude,longitude
0,138.842273,1773.659624,0,0,23.531145,87.369233
1,3.335142,1802.827634,0,0,23.505959,87.364391
2,8.135766,1782.467861,0,0,23.525812,87.363001
3,122.349736,911.772739,0,2,23.525331,87.379017
4,1.233551,681.101651,0,6,23.530724,87.334349


In [6]:
dataset_path, summary_path = save_model_training_dataset(
    training_gdf,
    summary,
    settings.processed_data_dir,
    settings.reports_dir,
)
dataset_path, summary_path


2026-04-28 15:40:49,014 | INFO | src.feature_engineering | Saved model training dataset to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\data\processed\model_training_dataset.parquet
2026-04-28 15:40:49,014 | INFO | src.feature_engineering | Saved feature summary to \\wsl.localhost\Ubuntu\home\pulkitv52\valuation_poc\reports\feature_summary.json


(WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/data/processed/model_training_dataset.parquet'),
 WindowsPath('//wsl.localhost/Ubuntu/home/pulkitv52/valuation_poc/reports/feature_summary.json'))

In [7]:
json.loads(Path(summary_path).read_text())


{'input_row_count': 273587,
 'output_row_count': 273587,
 'projected_crs': '{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "WGS 84 / UTM zone 45N", "base_crs": {"name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degre